# SBV2-KR 라이브러리 사용 예

이 저장소를 pip로 설치해 Jupyter Notebook에서 음성을 합성하는 예입니다. Google Colab 등에서도 동작합니다(Colab은 런타임 버전을 Python 3.12 이하로 지정).

PyPI의 `style-bert-vits2`는 업스트림 패키지라 한국어(KO)를 지원하지 않으므로 이 저장소에서 직접 설치합니다.

In [ ]:
# PyTorch 환경 구축 (없는 경우)
# 참고: https://pytorch.org/get-started/locally/

!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

In [ ]:
# SBV2-KR 설치. kiwipiepy는 한국어 형태소 기반 발음 규칙에 쓰이며, 없으면 일부 규칙이 꺼져 학습 때와 발음이 달라진다

!pip install git+https://github.com/ohgi07/SBV2-KR.git "kiwipiepy>=0.22,<0.24"

In [ ]:
# BERT 모델 로드 (직접 내려받을 필요 없음)

from style_bert_vits2.nlp import bert_models
from style_bert_vits2.constants import Languages


bert_models.load_model(Languages.JP, "ku-nlp/deberta-v2-large-japanese-char-wwm")
bert_models.load_tokenizer(Languages.JP, "ku-nlp/deberta-v2-large-japanese-char-wwm")
# bert_models.load_model(Languages.EN, "microsoft/deberta-v3-large")
# bert_models.load_tokenizer(Languages.EN, "microsoft/deberta-v3-large")
# bert_models.load_model(Languages.ZH, "hfl/chinese-roberta-wwm-ext-large")
# bert_models.load_tokenizer(Languages.ZH, "hfl/chinese-roberta-wwm-ext-large")

In [ ]:
# 공개된 한국어 모델이 없으므로 Hugging Face의 일본어 기본 모델을 내려받아 시험해 본다
# model_assets 디렉터리에 저장된다

from pathlib import Path
from huggingface_hub import hf_hub_download


model_file = "jvnv-F1-jp/jvnv-F1-jp_e160_s14000.safetensors"
config_file = "jvnv-F1-jp/config.json"
style_file = "jvnv-F1-jp/style_vectors.npy"

for file in [model_file, config_file, style_file]:
    print(file)
    hf_hub_download("litagin/style_bert_vits2_jvnv", file, local_dir="model_assets")

In [ ]:
# 위에서 내려받은 모델 파일을 지정해 음성 합성 시험

from style_bert_vits2.tts_model import TTSModel

assets_root = Path("model_assets")

model = TTSModel(
    model_path=assets_root / model_file,
    config_path=assets_root / config_file,
    style_vec_path=assets_root / style_file,
    device="cpu",
)

In [ ]:
from IPython.display import Audio, display

sr, audio = model.infer(text="こんにちは")
display(Audio(audio, rate=sr))

## 한국어 모델로 합성

직접 학습한 한국어 모델(학습 결과 `model_assets/{모델명}/`의 파일)을 지정하고, 한국어 BERT를 로드한 뒤 `language=Languages.KO`로 합성합니다.

In [ ]:
bert_models.load_model(Languages.KO, "klue/roberta-large")
bert_models.load_tokenizer(Languages.KO, "klue/roberta-large")

ko_model = TTSModel(
    model_path=assets_root / "your_model/your_model_e100_s10000.safetensors",
    config_path=assets_root / "your_model/config.json",
    style_vec_path=assets_root / "your_model/style_vectors.npy",
    device="cpu",
)

sr, audio = ko_model.infer(text="안녕하세요, 만나서 반가워요.", language=Languages.KO)
display(Audio(audio, rate=sr))